# Influential outliers comparison with BART

The goal of this notebook is to compare our influential outlier metric with the work by Pratola et al. and their Cook's distiance for Bayesian regression trees. We find similar performance on a simple data set with obvious influential outliers.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import shap

from sklearn.ensemble import RandomForestRegressor

In [ ]:
from main_iom import *

## BART influence metrics

We first read in the data and plot it.

In [ ]:
Data = pd.read_csv("Data_BART.csv").drop("Unnamed: 0", axis=1)
Data = Data.rename({"V1": 'x', "V2": "y"}, axis=1)
sns.scatterplot(Data, x='x', y='y')
plt.scatter(Data['x'][0], Data['y'][0], marker="*", s=150)
plt.scatter(Data['x'][1], Data['y'][1], marker="*", s=150) 


The BART model was fit using the `Ropenbt` package in `R`. The code for this can be found on Pratola's website https://bitbucket.org/mpratola/openbt/wiki/Home. We read in the regression curve by BART as well as the influence statistics from `R` and join them.

In [ ]:
fitp = pd.read_csv("BART_fit.csv", sep="\t", header=None)
fitp = fitp[fitp[0]<=1]
fitp_wo = pd.read_csv("BART_fit_without_out.csv")
fitp_wo = fitp_wo[fitp_wo['x']<=1]
cook_d = pd.read_csv("BART_cd.csv", sep="\t")
Cook = pd.concat([Data, cook_d], axis=1)

We first plot the regression curve predicted by BART.

In [ ]:
sns.scatterplot(Data, x='x', y='y')
sns.lineplot(fitp, x=0, y=1, label="With outliers")
sns.lineplot(fitp_wo, x='x', y='y', label="Without outliers")
plt.scatter(Data['x'][0], Data['y'][0], marker="*", s=150)
plt.scatter(Data['x'][1], Data['y'][1], marker="*", s=150)    

Now for the influence measurements, the $2\sigma$ and $3\sigma$ cutoff calculated as follows. 

In [ ]:
ref_cookd_2sd = 1/8*2**2*10/9**2
ref_cookd_3sd = 1/8*3**2*10/9**2

And we plot them.

In [ ]:
sns.scatterplot(Cook, x='x', y='MeanCookD')
plt.scatter(0.5, 0.226063, marker="*", s=150)  
plt.scatter(1,  0.522991, marker="*", s=150)  
plt.axhline(y=ref_cookd_2sd, color='green')
plt.axhline(y=ref_cookd_3sd, color='red')

In [ ]:
sns.scatterplot(Cook, x='x', y='MaxCookD')

plt.scatter(0.5, 0.857896, marker="*", s=150)  
plt.scatter(1,  2.860390, marker="*", s=150) 
plt.axhline(y=ref_cookd_2sd, color='green')
plt.axhline(y=ref_cookd_3sd, color='red')

## BART IOM

We test the IOM on BART using the `stochtree`. We trick `TreeExplainer` to take the BART model.

In [ ]:
from collections import deque

In [ ]:
from stochtree import BARTModel

bart = BARTModel()
bart.sample(X_train=Data[['x']], y_train=Data[['y']].values, num_gfr=10, num_mcmc=1000)


In [ ]:
import numpy as np
import shap
from collections import deque

# =============================================================
# Extract one stochtree tree as a dict for SingleTree
# =============================================================
def _extract_one_tree(fc, sample_idx, tree_idx, X_train):
    old_ids = []
    queue = deque([0])
    visited = set()
    while queue:
        nid = queue.popleft()
        if nid in visited:
            continue
        visited.add(nid)
        old_ids.append(nid)
        if not fc.is_leaf_node(sample_idx, tree_idx, nid):
            queue.append(fc.left_child_node(sample_idx, tree_idx, nid))
            queue.append(fc.right_child_node(sample_idx, tree_idx, nid))
    
    old_to_new = {old: new for new, old in enumerate(old_ids)}
    n = len(old_ids)
    
    children_left = np.full(n, -1, dtype=np.int32)
    children_right = np.full(n, -1, dtype=np.int32)
    features = np.full(n, -2, dtype=np.int32)
    thresholds = np.full(n, -2.0, dtype=np.float64)
    values = np.zeros((n, 1), dtype=np.float64)
    
    for old_id in old_ids:
        new_id = old_to_new[old_id]
        if fc.is_leaf_node(sample_idx, tree_idx, old_id):
            lv = fc.node_leaf_values(sample_idx, tree_idx, old_id)
            values[new_id, 0] = lv[0]
        else:
            left_old = fc.left_child_node(sample_idx, tree_idx, old_id)
            right_old = fc.right_child_node(sample_idx, tree_idx, old_id)
            children_left[new_id] = old_to_new[left_old]
            children_right[new_id] = old_to_new[right_old]
            features[new_id] = fc.node_split_index(sample_idx, tree_idx, old_id)
            thresholds[new_id] = fc.node_split_threshold(sample_idx, tree_idx, old_id)
    
    node_sample_weight = np.zeros(n, dtype=np.float64)
    for i in range(X_train.shape[0]):
        node = 0
        while children_left[node] != -1:
            node_sample_weight[node] += 1.0
            if X_train[i, features[node]] <= thresholds[node]:
                node = children_left[node]
            else:
                node = children_right[node]
        node_sample_weight[node] += 1.0
    
    return {
        "children_left": children_left,
        "children_right": children_right,
        "children_default": children_left.copy(),
        "features": features,
        "thresholds": thresholds,
        "values": values,
        "node_sample_weight": node_sample_weight,
    }


def stochtree_to_shap(bart_model, X_train, sample_idx=-1):
    fc = bart_model.forest_container_mean
    if sample_idx < 0:
        sample_idx = fc.num_samples() + sample_idx
    
    n_trees = fc.num_trees
    
    tree_dicts = []
    for tree_idx in range(n_trees):
        td = _extract_one_tree(fc, sample_idx, tree_idx, X_train)
        tree_dicts.append(td)
    
    model_dict = {
        "trees": tree_dicts,
        "base_offset": 0.0,
        "tree_output": "raw_value",
        "objective": "squared_error",
        "input_dtype": np.float64,
        "internal_dtype": np.float64,
    }
    
    return model_dict


# =============================================================
# Predict by manually walking trees (for verification)
# =============================================================
def predict_from_trees(tree_dicts, X):
    """Predict by summing leaf values across all trees."""
    n_obs = X.shape[0]
    preds = np.zeros(n_obs)
    
    for td in tree_dicts:
        cl = td["children_left"]
        cr = td["children_right"]
        feat = td["features"]
        thresh = td["thresholds"]
        vals = td["values"]
        
        for i in range(n_obs):
            node = 0
            while cl[node] != -1:
                if X[i, feat[node]] <= thresh[node]:
                    node = cl[node]
                else:
                    node = cr[node]
            preds[i] += vals[node, 0]
    
    return preds


# =============================================================
# Run it
# =============================================================
X_train = Data[['x']].values

fc = bart.forest_container_mean
sample_idx = fc.num_samples() - 1

print("Converting stochtree → shap model dict...")
model_dict = stochtree_to_shap(bart, X_train, sample_idx)
print(f"  {len(model_dict['trees'])} trees")

print("Creating TreeExplainer...")
explainer = shap.TreeExplainer(model_dict)

print("Computing SHAP values...")
shap_values_bart = explainer.shap_values(X_train)
print(f"  Shape: {shap_values_bart.shape}")

# Verify using our own tree prediction
pred = predict_from_trees(model_dict["trees"], X_train)
baseline = pred.mean()

print("\nVerification: sum(SHAP) ≈ f(x) - E[f(x)]")
for i in range(10):
    target = pred[i] - baseline
    got = shap_values[i].sum()
    err = abs(target - got)
    print(f"  Obs {i}: target={target:+.8f}  "
          f"sum={got:+.8f}  err={err:.2e} "
          f"{'✓' if err < 1e-6 else '✗'}")

# Also verify predictions match stochtree's predict
bart_pred = bart.predict(X_train)['y_hat'].mean(axis=0)  # posterior mean
our_pred = pred  # single posterior draw
print(f"\nOur tree prediction range: [{our_pred.min():.4f}, {our_pred.max():.4f}]")
print(f"BART posterior mean range: [{bart_pred.min():.4f}, {bart_pred.max():.4f}]")

In [ ]:
resid_bart = Data['y'].values - bart.predict(Data[['x']].values)['y_hat'].mean()

In [ ]:
print(f"SHAP values: {stats.shapiro(shap_values_bart.reshape(-1)).pvalue:.4f}")
print(f"Residuals: {stats.shapiro(resid_bart.reshape(-1)).pvalue:.4f}")

In [ ]:
iom_bart = InfluentialOutlierMetric(shap_values_bart.reshape(-1,1), resid_bart,
                                    1, 1, 1, 
                                    2, 1, 2,
                                    lambdas=np.concatenate([[0], np.exp(np.linspace(-1, 5, 50))]),
                                    lambdas_resid=np.concatenate([[0], np.exp(np.linspace(-1, 5, 50))]),
                                    epoch=500, epoch_resid=500)

In [ ]:
# Tune for lambda
iom_bart.find_best_lambda(alpha=0.05)
iom_bart.find_best_lambda_resid(alpha=0.05)

# Compute thresholds
thresholds = iom_bart.find_threshold(alpha=[0.05, 0.01])

# Compute IOM
IOM_bart = iom_bart.IOM()

In [ ]:
iom_bart.summary()

In [ ]:
Data['IOM'] = IOM_bart

In [ ]:
sns.scatterplot(Data, x='x', y='IOM')
plt.axhline(y=4.7609, color='green')
plt.axhline(y=12.9904, color='red')
plt.scatter(0.5, Data.loc[0,'IOM'], marker="*", s=150)
plt.scatter(1, Data.loc[1, "IOM"], marker="*", s=150) 

## Random Forest and IOM

We fit a standard random forest from `scikit-learn`. 

In [ ]:
clf = RandomForestRegressor(max_depth=2, random_state=0)
clf.fit(Data['x'].to_numpy().reshape(-1,1), Data['y'])

And we plot the estimated regression curve.

In [ ]:
clf_2 = RandomForestRegressor(max_depth=2, random_state=0)
clf_2.fit(Data['x'].drop([0,1]).to_numpy().reshape(-1,1), Data['y'].drop([0,1]))

In [ ]:
Pred = pd.concat([fitp[0], pd.Series(clf.predict(fitp[0].to_numpy().reshape(-1,1)))], axis=1, ignore_index=True)

In [ ]:
Pred_2 = pd.concat([fitp[0].drop([0,1]), pd.Series(clf_2.predict(fitp[0].drop([0,1]).to_numpy().reshape(-1,1)))], axis=1, ignore_index=True)

In [ ]:
sns.scatterplot(Data, x="x", y="y")
sns.lineplot(Pred, x=0, y=1, label='With outliers')
sns.lineplot(Pred_2, x=0, y=1, label='Without outliers')
plt.scatter(0.5, 8.157989, marker="*", s=150)
plt.scatter(1, 9.996990, marker="*", s=150) 

We calculate the residuals.

In [ ]:
resid = np.array(Data['y'] - clf.predict(Data['x'].to_numpy().reshape(-1,1)))

And using the `shap` package we calculate the SHAP values.

In [ ]:
explainer_rf = shap.TreeExplainer(clf) 
shap_values_rf = explainer_rf.shap_values(Data['x'].to_numpy().reshape(-1,1)) 

We assess normality.

In [ ]:
print(f"SHAP values: {stats.shapiro(shap_values_rf.reshape(-1)).pvalue:.4f}")
print(f"Residuals: {stats.shapiro(resid.reshape(-1)).pvalue:.4f}")

In [ ]:
iom_rf = InfluentialOutlierMetric(shap_values_rf.reshape(-1,1), resid,
                                    2, 2, 2, 
                                    2, 2, 2, 
                                    lambdas=np.concatenate([[0], np.exp(np.linspace(-3, 5, 50))]),
                                    lambdas_resid=np.concatenate([[0], np.exp(np.linspace(-4, 5, 50))]),
                                    epoch=500, epoch_resid=500)

In [ ]:
# Tune for lambda
iom_rf.find_best_lambda(alpha=0.05)
iom_rf.find_best_lambda_resid(alpha=0.05)

# Compute thresholds
thresholds = iom_rf.find_threshold(alpha=[0.05, 0.01])

# Compute IOM
IOM_rf = iom_rf.IOM()

In [ ]:
iom_rf.summary()

In [ ]:
Data['IOM'] = IOM_rf

In [ ]:
sns.scatterplot(Data, x='x', y='IOM')
plt.axhline(y=4.7609, color='green')
plt.axhline(y=12.9904, color='red')
plt.scatter(0.5, Data.loc[0,'IOM'], marker="*", s=150)
plt.scatter(1, Data.loc[1,'IOM'], marker="*", s=150) 